In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Paths
DATASET_ROOT = r"E:\404_Found\H&E Stained Histological Nucleus Instancve Segmentation by YOLOv8x\Dataset\PanNuke_Dataset_Splitted"  

SPLITS = ["train", "val"]  # NO test

def mask_to_yolo(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    lines = []

    for cnt in contours:
        if len(cnt) < 3:
            continue

        epsilon = 0.002 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        points = approx.reshape(-1, 2)

        if len(points) < 3:
            continue

        # normalize
        norm = [(x / img_w, y / img_h) for x, y in points]

        flat = []
        for x, y in norm:
            flat.append(f"{x:.6f}")
            flat.append(f"{y:.6f}")

        line = "0 " + " ".join(flat)
        lines.append(line)

    return lines


for split in SPLITS:
    print(f"Processing {split}...")

    img_dir = os.path.join(DATASET_ROOT, split, "images")
    mask_dir = os.path.join(DATASET_ROOT, split, "labels")

    for img_name in tqdm(os.listdir(img_dir)):
        if not img_name.endswith(".png"):
            continue

        img_path = os.path.join(img_dir, img_name)
        mask_path = os.path.join(mask_dir, img_name)

        if not os.path.exists(mask_path):
            continue

        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        h, w = mask.shape

        # ensure binary
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        yolo_lines = mask_to_yolo(mask, w, h)

        # overwrite label file as .txt
        txt_path = os.path.join(mask_dir, img_name.replace(".png", ".txt"))

        with open(txt_path, "w") as f:
            f.write("\n".join(yolo_lines))

    print(f"{split} done.")

print("ALL DONE")

Processing train...


100%|██████████| 5530/5530 [02:46<00:00, 33.23it/s]


train done.
Processing val...


100%|██████████| 1185/1185 [00:39<00:00, 29.83it/s]

val done.
ALL DONE
